[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/process_tomography/process_tomography_bloch.ipynb)

# Quantum Process Tomography with Bloch Vectors

Quantum process tomography recovers a quantum device's channel on every qubit state from outcome probabilities measured on known input states. In the conventional Pauli-algebra picture, a qubit state is modelled as half the unit scalar plus half its three-dimensional Bloch vector, so pure states have unit Bloch vectors and mixed states have shorter Bloch vectors down to zero at the completely mixed state. Four input states arranged as a regular tetrahedron span the four-dimensional scalar-plus-vector state space and determine the channel everywhere.

Keeping the scalar normalization part inside `State` makes even non-unital relaxation a linear map `State <- State`, built from operator sandwiches such as `loss_paths * State * ~loss_paths`. Detector outcome probabilities come from the scalar product `2 * effects.scalar_product(outputs)`, and solving the symmetric overlap map `(states * states.scalar_product(State)).sum(axis=-1)` produces reciprocal dual frames that assemble the reconstructed channel directly as `(recovered_outputs * preparation_dual.scalar_product(State)).sum(axis=-1)`. See [the projective process tomography notebook](process_tomography.ipynb) for the metric-free projective formulation, following [Chuang and Nielsen](https://arxiv.org/abs/quant-ph/9610001) and the symmetric tetrahedral measurement of [Renes et al.](https://arxiv.org/abs/quant-ph/0310075).

In density-matrix and $\chi$-matrix process tomography, a state reads as $\rho = \tfrac12(I + \mathbf r \cdot \boldsymbol\sigma)$ and a channel is expanded in a fixed operator basis $\{E_m\}$ as $\mathcal E(\rho) = \sum_{m,n} \chi_{mn} E_m \rho E_n^\dagger$, requiring a generalized inverse of the fourth-rank tensor $\beta^{mn}_{jk}$ in $\sum_{m,n} \beta^{mn}_{jk}\,\chi_{mn} = \lambda_{jk}$ to recover $\chi_{mn}$ from state-expansion coefficients $\mathcal E(\rho_j) = \sum_k \lambda_{jk} \rho_k$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.quantum.process_tomography import render
from examples.util.animation import show_states

np.set_printoptions(precision=4, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Bivector = ga.gatype.bivector()                                              # Bivector+
# A state is a scalar plus a vector, fixed by reversal; a channel maps this four-dimensional space.
State = ga.gatype.self_reverse()                                             # Scalar+ + Vector+
Channel = ga.gatype((State, State))                                          # State+ <- State+
one = mv.scalar([1])                                                         # [] Scalar+
north = (one + mv.z) / 2                                                     # [] State+
south = (one - mv.z) / 2                                                     # [] State+

## Preparing, evolving, and measuring

Four pure states pointing along the corners of a regular tetrahedron pass through each gate. Four detector effects along the same directions sum to `one`, and each outcome probability equals twice the scalar product of the effect with the outgoing state. Unobserved noise paths act by summing their individual operator sandwiches on `State`, which automatically produces the `sqrt(1 - loss)` transverse coherence factor under amplitude damping.

In density-matrix notation, the POVM elements read as $F_b = \tfrac14(I + \mathbf n_b \cdot \boldsymbol\sigma)$ and the Born probabilities as $p(b \mid a) = \operatorname{tr}[F_b\,\mathcal E(\rho_a)]$.

In [ ]:
labels = ("Rotation", "Rotation + dephasing", "Rotation + relaxation")
angles = np.array([0.075, 0.075, 0.075])
phase_flip = np.array([0.0, 0.06, 0.0])
loss = np.array([0.0, 0.0, 0.10])
rotation_plane = (mv.yz + mv.zx + mv.xy) / np.sqrt(3)                        # [] Bivector+

directions = mv.vector([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]) / np.sqrt(3)  # [4] Vector+
preparations = (one + directions) / 2                                        # [4] State+
effects = preparations / 2                                                   # [4] State+

In [ ]:
def noisy_gate(
    angles: np.ndarray, phase_flip: np.ndarray, loss: np.ndarray, rotation_plane: Bivector,
) -> Channel:
    """Kraus phase-noise and relaxation sandwiches followed by a coherent rotor form a gate channel (`[...] State+ <- State+`)."""
    # Each unobserved phase-noise branch contributes its own rotor sandwich weighted by probability.
    phase_paths = stack((one * np.sqrt(1 - phase_flip), mv.xy * np.sqrt(phase_flip)), axis=-1)  # [..., 2] Rotor+
    dephasing = (phase_paths >> State).sum(axis=-1)                          # [...] State+ <- State+
    # One amplitude-damping branch attenuates south; the mixed-parity jump mv.x * south transfers south to north.
    loss_paths = stack(
        (north + south * np.sqrt(1 - loss), (mv.x * south) * np.sqrt(loss)), axis=-1,
    )                                                                        # [..., 2] Scalar+ + Vector+ + Bivector+
    damping = (loss_paths * State * ~loss_paths).cast(Channel).sum(axis=-1)  # [...] State+ <- State+
    rotations = (rotation_plane * (-angles / 2)).exp()                       # [...] Rotor+
    return rotations >> damping(dephasing)                                   # [...] State+ <- State+


def outcome_probabilities(channel: Channel, states: State, effects: State) -> Scalar:
    """Twice the scalar product of `effects` with mapped `states` gives the Born probabilities (`[...] Scalar+`)."""
    outputs = channel[..., None](states)                                     # [...] State+
    return 2 * effects.scalar_product(outputs[..., None])                    # [...] Scalar+


true_channels = noisy_gate(angles, phase_flip, loss, rotation_plane)         # [3] State+ <- State+
measurements = outcome_probabilities(true_channels, preparations, effects)   # [3, 4, 4] Scalar+
render.experiment_figure(preparations, measurements, labels, render.bloch_positions);

The left panel places the Bloch-vector tips of the four tetrahedral preparations `A`, `B`, `C`, and `D` on the unit sphere, and the three log-scaled heatmaps show the sixteen outcome probabilities for pure rotation, dephasing, and relaxation. Diagonal entries stay brightest under the small coherent turn, while noise raises the off-diagonal probabilities.

## Recovering the channel

Because the four tetrahedral states are non-orthogonal under `scalar_product`, raw projections mix their contributions. Solving the frame operator `(states * states.scalar_product(State)).sum(axis=-1)` produces the dual frame that unmixes them across the four-dimensional state space. Applying the detector's dual frame reconstructs the four outgoing states, and summing their dyads against the preparations' dual frame recovers the full `State <- State` map.

In tensor-index notation, recovering the process matrix $\chi_{mn}$ requires a generalized inverse $\kappa$ satisfying $\beta^{mn}_{jk} = \sum_{st,xy} \beta^{st}_{jk}\,\kappa^{xy}_{st}\,\beta^{mn}_{xy}$.

In [ ]:
n_latitudes = 12
n_longitudes = 2 * n_latitudes

In [ ]:
def dual_frame(states: State) -> State:
    """The inverse of the scalar-product overlap map gives the reciprocal dual elements (`[...] State+`)."""
    overlap = (states * states.scalar_product(State)).sum(axis=-1)           # [...] State+ <- State+
    return overlap[..., None].solve(states)                                  # [...] State+


def reconstructed_channel(preparations: State, effects: State, measurements: Scalar) -> Channel:
    """Reciprocal dual frames of `effects` and `preparations` recover the linear channel (`[...] State+ <- State+`)."""
    measurement_dual = dual_frame(2 * effects)                               # [4] State+
    recovered_outputs = (measurements * measurement_dual).sum(axis=-1)       # [..., 4] State+
    preparation_dual = dual_frame(preparations)                              # [4] State+
    return (recovered_outputs * preparation_dual.scalar_product(State)).sum(axis=-1)  # [...] State+ <- State+

In [ ]:
def pure_sphere(n_latitudes: int, n_longitudes: int) -> State:
    """Inclination tilts and azimuthal turns sweep `north` across a closed sphere grid (`[n_latitudes + 1, n_longitudes] State+`)."""
    inclinations = np.linspace(0, np.pi, n_latitudes + 1)
    azimuths = np.linspace(0, 2 * np.pi, n_longitudes, endpoint=False)
    tilts = (mv.zx * (-inclinations / 2)).exp()                              # [n_latitudes + 1] Rotor+
    turns = (mv.xy * (-azimuths / 2)).exp()                                  # [n_longitudes] Rotor+
    meridian = tilts >> north                                                # [n_latitudes + 1] State+
    return turns[None, :] >> meridian[:, None]                               # [n_latitudes + 1, n_longitudes] State+


learned_channels = reconstructed_channel(preparations, effects, measurements)  # [3] State+ <- State+
surface = pure_sphere(n_latitudes, n_longitudes)                             # [n_latitudes + 1, n_longitudes] State+
sphere_images = learned_channels[:, None, None](surface)                     # [3, n_latitudes + 1, n_longitudes] State+
probe_images = learned_channels[:, None](preparations)                       # [3, 4] State+
render.channels_figure(surface, sphere_images, preparations, probe_images, labels, render.bloch_positions);

Each panel overlays the translucent unit sphere of pure inputs and the four larger translucent prepared probes with the reconstructed output surface and smaller solid output probes, matching the projective reconstruction identically across rotation, dephasing, and relaxation.

## Predicting unseen inputs

Because the channel acts linearly on `State`, the recovered map predicts detector probabilities across pure and mixed states absent from the calibration set. Relaxation couples the scalar normalization component into the `z` vector component, shifting the completely mixed state toward `north` while dephasing leaves the centre fixed.

In Bloch-vector notation, the channel acts as the affine map $\mathbf r' = A\mathbf r + \mathbf t$, separating the linear contraction $A$ from the centre shift $\mathbf t$.

In [ ]:
validation_directions = mv.vector(
    [[1, 0, 0], [0, 1, 0], [0, 0, 1], [-1, 0, 0], [0, -1, 0], [0, 0, -1]],
)                                                                            # [6] Vector+
bloch_lengths = np.array([1.0, 0.7, 0.3, 1.0, 0.7, 0.0])

In [ ]:
validation_states = (one + validation_directions * bloch_lengths) / 2        # [6] State+
observed_probabilities = outcome_probabilities(true_channels, validation_states, effects)      # [3, 6, 4] Scalar+
predicted_probabilities = outcome_probabilities(learned_channels, validation_states, effects)  # [3, 6, 4] Scalar+
render.predictions_figure(observed_probabilities, predicted_probabilities, labels);

All twenty-four state-outcome probabilities for each of the three channels fall on the diagonal equality line, confirming exact recovery across pure, partially mixed, and completely mixed validation states.

## Spanning preparations

Two opposite preparations probe only the scalar component and a single spin axis, leaving channels that scramble the two transverse directions indistinguishable on them. Comparing the singular values of the overlap map shows four positive singular values for the tetrahedral probes and two vanishing singular values for opposite probes.

In [ ]:
design_labels = ("Four tetrahedral probes", "Two opposite probes")

In [ ]:
tetrahedral_overlap = (preparations * preparations.scalar_product(State)).sum(axis=-1)  # [] State+ <- State+
opposite_probes = stack((north, south))                                      # [2] State+
opposite_overlap = (opposite_probes * opposite_probes.scalar_product(State)).sum(axis=-1).cast(Channel)  # [] State+ <- State+
spectra = stack((tetrahedral_overlap.svdvals(), opposite_overlap.svdvals()))  # [2, 4] Scalar+

for design_label, spectrum in zip(design_labels, spectra):
    print(f"{design_label}: {spectrum.to_array()}")

## Predicting repeated uses

Composing the reconstructed `State <- State` map with itself predicts any number of successive passes through the gate without further calibration measurements.

In superoperator notation, $n$ successive applications of the gate read as $\widehat{\mathcal E}^{\,n}$.

In [ ]:
n_uses = 48
frame_duration_ms = 40

In [ ]:
def channel_powers(channel: Channel, n_uses: int) -> Iterator[Channel]:
    """Yields the composed channels (`[...] State+ <- State+`) for one through `n_uses` applications of `channel`."""
    channel_power = channel                                                  # [...] State+ <- State+
    for _ in range(n_uses):
        yield channel_power
        channel_power = channel(channel_power)                               # [...] State+ <- State+


frames = (
    (channel_power[:, None, None](surface), channel_power[:, None](preparations))
    for channel_power in channel_powers(learned_channels, n_uses)
)
power_frames = show_states(
    frames, render.channel_animation, surface, preparations, labels, render.bloch_positions, render.panel_size,
    name="process_tomography_bloch", duration_ms=frame_duration_ms,
);

Across repeated passes, pure rotation winds the four solid output probes steadily around the `(1, 1, 1)` axis on the unit sphere, dephasing spirals the ellipsoid inward toward the tilted rotation axis near the centre, and relaxation draws the contracting ellipsoid toward a fixed point near the north pole.

In [ ]:
# checks
assert (measurements >= -1e-12).all()
assert ((measurements.sum(axis=-1) - 1).abs() < 1e-12).all()
assert ((outcome_probabilities(learned_channels, preparations, effects) - measurements).abs() < 1e-12).all()
assert ((observed_probabilities - predicted_probabilities).abs() < 1e-12).all()
assert (spectra[0] > 0.1).all()
assert (spectra[1, :2] > 0.1).all()
assert (spectra[1, 2:].abs() < 1e-12).all()
assert len(power_frames) == n_uses
final_spheres, final_probes = power_frames[-1]
assert (((2 * final_probes.select.scalar()) - 1).abs() < 1e-10).all()
assert (((2 * final_spheres[0].select.vector()).norm() - 1).abs() < 1e-10).all()
assert ((2 * final_spheres[1].select.vector()).norm() < 0.4).all()
assert ((2 * final_spheres[2].select.vector()).norm() < 0.99).all()
assert ((final_spheres[2] - north).scalar_norm_squared() < (final_spheres[2] - south).scalar_norm_squared()).all()